# Classification Practical Session (90 minutes)

**Learning Objectives:**
- Build and compare three classification algorithms: KNN, Decision Trees, and Logistic Regression
- Tune hyperparameters using GridSearchCV
- Evaluate models using multiple metrics
- Select optimal decision thresholds
- Interpret model predictions

**What You'll Do:**
1. Load and explore the Breast Cancer dataset
2. Train baseline classifiers
3. Tune hyperparameters to improve performance
4. Compare models and select the best one
5. Optimize the decision threshold
6. Interpret feature importance

**Structure:**
- ✅ Pre-filled: Data loading, preprocessing, and utility functions
- ✏️ Your work: Complete TODO sections to build and analyze classifiers

> **Tip:** Work through sections sequentially. Each TODO builds on previous work.

## 1) Setup & Imports (⏱️ ~2 min)

In [ ]:
# If needed:
# !pip install -q numpy pandas scikit-learn matplotlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV, learning_curve, validation_curve
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, confusion_matrix,
    RocCurveDisplay, PrecisionRecallDisplay
)
from sklearn.calibration import CalibrationDisplay
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.linear_model import LogisticRegression
from sklearn.inspection import permutation_importance

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)


## 2) Load Dataset (⏱️ ~3 min)

In [ ]:
from sklearn.datasets import load_breast_cancer

def load_default_data():
    data = load_breast_cancer()
    X = pd.DataFrame(data.data, columns=data.feature_names)
    y = pd.Series(data.target, name='target')
    return pd.concat([X, y], axis=1), 'target'

df, target_col = load_default_data()
df.head(3)


## 3) Quick Data Exploration (⏱️ ~3 min)

In [ ]:
print('Shape:', df.shape)
print('\nMissing values (top 10):')
print(df.isna().sum().sort_values(ascending=False).head(10))
print('\nClass balance:')
print(df[target_col].value_counts(normalize=True).rename('proportion'))
df.describe().T.head(8)


## 4) Data Preprocessing (⏱️ ~2 min)

In [ ]:
feature_cols = [c for c in df.columns if c != target_col]
X = df[feature_cols]
y = df[target_col]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=RANDOM_STATE
)

numeric_features = feature_cols
preprocess = ColumnTransformer([
    ('num', StandardScaler(), numeric_features),
], remainder='drop')

print('Train:', X_train.shape, ' Test:', X_test.shape)


## 5) Evaluation Utilities (pre-filled — run this cell)

In [ ]:
def evaluate_classifier(y_true, y_score, threshold=0.5):
    """Compute classification metrics at given threshold"""
    y_pred = (y_score >= threshold).astype(int)
    metrics = {
        'accuracy': accuracy_score(y_true, y_pred),
        'precision': precision_score(y_true, y_pred, zero_division=0),
        'recall': recall_score(y_true, y_pred, zero_division=0),
        'f1': f1_score(y_true, y_pred, zero_division=0),
        'roc_auc': roc_auc_score(y_true, y_score),
        'pr_auc': average_precision_score(y_true, y_score),
    }
    return metrics, y_pred

def plot_confusion_matrix(cm, labels=(0,1)):
    """Visualize confusion matrix"""
    fig, ax = plt.subplots(figsize=(4,4))
    ax.imshow(cm, interpolation='nearest', cmap='Blues')
    ax.set_title('Confusion Matrix')
    ax.set_xticks([0,1]); ax.set_yticks([0,1])
    ax.set_xticklabels(labels); ax.set_yticklabels(labels)
    for (i, j), v in np.ndenumerate(cm):
        ax.text(j, i, str(v), ha='center', va='center', color='white' if v > cm.max()/2 else 'black')
    ax.set_xlabel('Predicted'); ax.set_ylabel('True')
    plt.show()

def plot_roc_pr(y_true, y_score):
    """Plot ROC and Precision-Recall curves"""
    RocCurveDisplay.from_predictions(y_true, y_score)
    plt.title('ROC Curve'); plt.show()
    PrecisionRecallDisplay.from_predictions(y_true, y_score)
    plt.title('Precision-Recall Curve'); plt.show()

def plot_probability_density(y_true, y_score):
    """Show distribution of predicted probabilities by true class"""
    y_true = np.asarray(y_true)
    pos = y_score[y_true == 1]
    neg = y_score[y_true == 0]
    fig, ax = plt.subplots(figsize=(6,4))
    ax.hist(neg, bins=20, alpha=0.6, density=True, label='Class 0 (Benign)')
    ax.hist(pos, bins=20, alpha=0.6, density=True, label='Class 1 (Malignant)')
    ax.set_title('Probability Density of Predictions')
    ax.set_xlabel('P(y=1)'); ax.set_ylabel('Density'); ax.legend()
    plt.show()

def calibration_plot(y_true, y_score, n_bins=10):
    """Check if predicted probabilities match observed frequencies"""
    fig, ax = plt.subplots(figsize=(5,4))
    CalibrationDisplay.from_predictions(y_true, y_score, n_bins=n_bins, ax=ax)
    ax.set_title('Calibration Curve'); plt.show()

def summarize_metrics(metrics_dict):
    """Create comparison table from metrics dictionary"""
    rows = []
    for name, m in metrics_dict.items():
        row = {'model': name}; row.update(m); rows.append(row)
    return pd.DataFrame(rows).set_index('model').sort_values(by='f1', ascending=False)

def find_best_threshold(y_true, y_score, metric='f1', num=201):
    """Find optimal classification threshold for given metric"""
    thresholds = np.linspace(0,1,num)
    best_t, best_v = 0.5, -1
    hist = []
    for t in thresholds:
        m, _ = evaluate_classifier(y_true, y_score, threshold=t)
        v = m[metric]
        hist.append((t, v))
        if v > best_v:
            best_t, best_v = t, v
    return best_t, best_v, pd.DataFrame(hist, columns=['threshold', metric])

def plot_validation_curve(estimator, X, y, param_name, param_range, scoring='f1'):
    """Plot validation curve showing train/test scores vs hyperparameter"""
    train_scores, test_scores = validation_curve(
        estimator, X, y, param_name=param_name, param_range=param_range,
        cv=5, scoring=scoring, n_jobs=-1
    )
    plt.figure(figsize=(8,5))
    plt.plot(param_range, test_scores.mean(axis=1), label='Validation', marker='o')
    plt.fill_between(param_range, test_scores.mean(axis=1) - test_scores.std(axis=1),
                     test_scores.mean(axis=1) + test_scores.std(axis=1), alpha=0.2)
    plt.plot(param_range, train_scores.mean(axis=1), label='Training', marker='o')
    plt.fill_between(param_range, train_scores.mean(axis=1) - train_scores.std(axis=1),
                     train_scores.mean(axis=1) + train_scores.std(axis=1), alpha=0.2)
    plt.xlabel(param_name.replace('clf__', '')); plt.ylabel(f'{scoring.upper()} Score')
    plt.legend(); plt.title(f'Validation Curve'); plt.grid(alpha=0.3)
    plt.show()

def plot_learning_curve(estimator, X, y, scoring='f1'):
    """Plot learning curve showing performance vs training set size"""
    train_sizes, train_scores, test_scores = learning_curve(
        estimator, X, y, cv=5, scoring=scoring,
        train_sizes=np.linspace(0.1, 1.0, 10), random_state=RANDOM_STATE, n_jobs=-1
    )
    plt.figure(figsize=(8,5))
    plt.plot(train_sizes, train_scores.mean(axis=1), label='Training', marker='o')
    plt.fill_between(train_sizes, train_scores.mean(axis=1) - train_scores.std(axis=1),
                     train_scores.mean(axis=1) + train_scores.std(axis=1), alpha=0.2)
    plt.plot(train_sizes, test_scores.mean(axis=1), label='Validation', marker='o')
    plt.fill_between(train_sizes, test_scores.mean(axis=1) - test_scores.std(axis=1),
                     test_scores.mean(axis=1) + test_scores.std(axis=1), alpha=0.2)
    plt.xlabel('Training Set Size'); plt.ylabel(f'{scoring.upper()} Score')
    plt.legend(); plt.title('Learning Curve'); plt.grid(alpha=0.3)
    plt.show()

print("✓ Utility functions loaded")

## 6) Build Baseline Models ✏️ TODO (⏱️ ~15 min)

**Your Task:**
Create three classification pipelines and evaluate their baseline performance.

**Requirements:**
1. Create a dictionary called `pipelines` with three models:
   - `'KNN'`: K-Nearest Neighbors with `n_neighbors=5`
   - `'DecisionTree'`: Decision Tree with `max_depth=5`
   - `'LogisticRegression'`: Logistic Regression with `max_iter=1000`

2. Each pipeline should include the `preprocess` step and the classifier

3. For each model:
   - Fit on `X_train, y_train`
   - Get probability predictions on `X_test` using the helper function `proba_1`
   - Evaluate at threshold=0.5 using `evaluate_classifier`
   - Store scores in `scores` dict and metrics in `metrics_at_05` dict

4. Create summary table: `summary_05 = summarize_metrics(metrics_at_05)`

**Questions to consider:**
- Which model performs best initially?
- What are the trade-offs between precision and recall?

In [ ]:
# TODO: Create pipelines dictionary
pipelines = {}

def proba_1(pipe, X):
    """Helper: extract probability for class 1"""
    return pipe.predict_proba(X)[:,1]

# TODO: Fit each pipeline and evaluate
metrics_at_05 = {}
scores = {}

# Your code here...


# TODO: Create summary table
# summary_05 = summarize_metrics(metrics_at_05)
# summary_05

## 7) Hyperparameter Exploration ✏️ TODO (⏱️ ~20 min)

### Part A: Validation Curves - Compute

**Your Task:** Understand how different hyperparameters affect model performance.

Compute validation curves (scoring='f1') for each model:
- **KNN**: `clf__n_neighbors` from 1 to 20
- **Decision Tree**: `clf__max_depth` from 1 to 20  
- **Logistic Regression**: `clf__C` over `np.logspace(-3, 3, 7)`

Use `validation_curve` from sklearn to compute train/test scores. Store results for plotting.

**Questions:**
- Where does each model overfit vs underfit?
- What hyperparameter values seem most promising?

In [ ]:
# TODO: Create validation curves for each model

# Your code here...

### Part A: Validation Curves - Visualize

Now plot the validation curves you just computed.

In [ ]:
# TODO: Plot validation curves

# Your code here...

### Part B: Learning Curves - Compute

**Your Task:** Analyze if models would benefit from more training data.

Compute learning curves (scoring='f1') for each of your three baseline pipelines.
Use `learning_curve` to get train sizes and scores.

**Questions:**
- Which models show high bias (underfitting)?
- Which show high variance (overfitting)?
- Would collecting more data help any of these models?

In [ ]:
# TODO: Plot learning curves for each baseline model

# Your code here...

### Part B: Learning Curves - Visualize

Now plot the learning curves.

In [ ]:
# TODO: Plot learning curves

# Your code here...

## 8) Hyperparameter Tuning ✏️ TODO (⏱️ ~20 min)

**Your Task:** Use GridSearchCV to find optimal hyperparameters for each model.

**Requirements:**

1. Define parameter grids for each model:
   - **KNN**: `clf__n_neighbors=[3,5,7,11,15]`, `clf__weights=['uniform','distance']`
   - **Decision Tree**: `clf__max_depth=[3,5,7,9]`, `clf__min_samples_leaf=[1,3,5,10]`
   - **Logistic Regression**: `clf__C=np.logspace(-2,2,5)`, `clf__penalty=['l2']`, `clf__solver=['lbfgs']`

2. For each model:
   - Use `GridSearchCV` with `scoring='f1'`, `cv=5`, `refit=True`
   - Fit on `X_train, y_train`
   - Store best estimator in `best_pipes[name]`
   - Save top 10 results in `gs_results[name]` dataframe

3. Inspect the top results from each grid search

**Questions:**
- Which hyperparameters had the biggest impact?
- Do you see signs of overfitting in train vs test scores?

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

# TODO: Define parameter grids
grids = {}

# Your code here...


# Storage for results
gs_results = {}
best_pipes = {}

# TODO: Run GridSearchCV for each model

# Your code here...


# TODO: Inspect top results for each model

## 9) Model Selection ✏️ TODO (⏱️ ~10 min)

**Your Task:** Compare tuned models and select the best one.

1. Evaluate all tuned models from `best_pipes` on `X_test`
2. Compute metrics at threshold=0.5 for each
3. Create `summary_tuned` table using `summarize_metrics`
4. Select the best model: `best_name = summary_tuned.index[0]`

**Questions:**
- How much did tuning improve performance?
- Which model architecture works best for this problem?

In [ ]:
# TODO: Evaluate all tuned models
metrics_tuned = {}
scores_tuned = {}

# Your code here...


# TODO: Create summary table and select best model
# summary_tuned = summarize_metrics(metrics_tuned)
# summary_tuned

# best_name = summary_tuned.index[0]
# best_name

### Diagnostic Analysis ✏️ TODO

**Your Task:** Analyze the best model's performance in detail.

**Step 1: Compute Metrics**
- Extract predictions for the best model
- Compute metrics and confusion matrix at threshold=0.5
- Print key statistics

**Step 2: Create Visualizations** (in next cell)
Generate diagnostic plots to understand model behavior.

In [ ]:
# TODO: Generate diagnostic plots for best model

# Your code here...

### Diagnostic Plots

Create the following visualizations:
1. Confusion matrix
2. ROC curve
3. Precision-Recall curve
4. Calibration plot
5. Probability density histogram

**Questions:**
- What types of errors is the model making?
- Is the model well-calibrated?
- How separable are the classes?

In [ ]:
# TODO: Create diagnostic plots

# Your code here...

## 10) Threshold Optimization ✏️ TODO (⏱️ ~10 min)

### Step 1: Find Optimal Threshold

**Your Task:** Find the optimal decision threshold for your best model.

1. Use `find_best_threshold` on `y_test` and `y_score_best` to optimize F1 score
2. Print the optimal threshold and compare with default (0.5)
3. Store the threshold value and history for plotting

**Questions:**
- How does the optimal threshold differ from 0.5?
- Why might we want to adjust the threshold in medical diagnosis?

In [ ]:
# TODO: Find optimal threshold and compare models

# Your code here...

### Step 2: Visualize Threshold Sweep

Plot the F1 score across different threshold values.

In [ ]:
# TODO: Plot threshold sweep

# Your code here...

### Step 3: Compare All Models

Re-evaluate ALL tuned models at the optimal threshold and create comparison table.

**Question:**
- Does the model ranking change with the new threshold?

In [ ]:
# TODO: Re-evaluate all models at optimal threshold

# Your code here...

## 11) Model Interpretation ✏️ TODO (⏱️ ~10 min)

**Your Task:** Understand which features drive predictions.

### Logistic Regression Coefficients
- Extract coefficients from the fitted LogisticRegression pipeline
- Show top 10-12 features by absolute coefficient value
- Interpret: positive coefficients push toward class 1 (malignant)

### Decision Tree Feature Importance
- Use `permutation_importance` on the fitted DecisionTree
- Show top 10-12 features by importance
- Compare with Logistic Regression insights

**Questions:**
- Which features are most important for prediction?
- Do the two models agree on feature importance?
- How do these findings align with medical knowledge?

In [ ]:
# TODO: Analyze feature importance

# Your code here...

---

## Summary & Reflection

**Congratulations!** You've completed a full classification workflow.

**Key Takeaways:**
- Different algorithms have different strengths and weaknesses
- Hyperparameter tuning can significantly improve performance
- The default threshold (0.5) isn't always optimal
- Model interpretation helps build trust and understanding

**Reflection Questions:**
1. Which model would you deploy and why?
2. What threshold would you choose for a medical diagnosis system?
3. What would you do differently with more time or data?
4. How confident are you in the model's predictions?

---

## 📚 Hints & Reference (Expand if stuck)

<details>
<summary><b>Section 6: Building Baseline Models</b></summary>

```python
from sklearn.pipeline import Pipeline

pipelines = {
    'KNN': Pipeline([
        ('prep', preprocess), 
        ('clf', KNeighborsClassifier(n_neighbors=5))
    ]),
    'DecisionTree': Pipeline([
        ('prep', preprocess), 
        ('clf', DecisionTreeClassifier(max_depth=5, random_state=RANDOM_STATE))
    ]),
    'LogisticRegression': Pipeline([
        ('prep', preprocess), 
        ('clf', LogisticRegression(max_iter=1000, random_state=RANDOM_STATE))
    ]),
}

for name, pipe in pipelines.items():
    pipe.fit(X_train, y_train)
    s = proba_1(pipe, X_test)
    scores[name] = s
    m, _ = evaluate_classifier(y_test, s, threshold=0.5)
    metrics_at_05[name] = m
```
</details>

<details>
<summary><b>Section 7A: Validation Curves</b></summary>

```python
from sklearn.model_selection import validation_curve

# Example for KNN
train_scores, test_scores = validation_curve(
    Pipeline([('prep', preprocess), ('clf', KNeighborsClassifier())]),
    X_train, y_train,
    param_name='clf__n_neighbors',
    param_range=np.arange(1, 21),
    cv=5, scoring='f1'
)

plt.plot(np.arange(1, 21), test_scores.mean(axis=1), label='Validation')
plt.plot(np.arange(1, 21), train_scores.mean(axis=1), label='Training')
plt.xlabel('n_neighbors'); plt.ylabel('F1 Score')
plt.legend(); plt.title('KNN Validation Curve'); plt.show()
```
</details>

<details>
<summary><b>Section 7B: Learning Curves</b></summary>

```python
from sklearn.model_selection import learning_curve

def plot_learning_curve(estimator, X, y, scoring='f1'):
    train_sizes, train_scores, test_scores = learning_curve(
        estimator, X, y, cv=5, scoring=scoring,
        train_sizes=np.linspace(0.1, 1.0, 10), random_state=RANDOM_STATE
    )
    plt.plot(train_sizes, train_scores.mean(axis=1), label='Training')
    plt.plot(train_sizes, test_scores.mean(axis=1), label='Validation')
    plt.xlabel('Training Size'); plt.ylabel(f'{scoring.upper()} Score')
    plt.legend(); plt.title('Learning Curve'); plt.show()

plot_learning_curve(pipelines['KNN'], X_train, y_train)
```
</details>

<details>
<summary><b>Section 8: GridSearchCV</b></summary>

```python
grids = {
    'KNN': {
        'clf__n_neighbors': [3, 5, 7, 11, 15],
        'clf__weights': ['uniform', 'distance']
    },
    'DecisionTree': {
        'clf__max_depth': [3, 5, 7, 9],
        'clf__min_samples_leaf': [1, 3, 5, 10]
    },
    'LogisticRegression': {
        'clf__C': np.logspace(-2, 2, 5),
        'clf__penalty': ['l2'],
        'clf__solver': ['lbfgs']
    }
}

for name, base in pipelines.items():
    gscv = GridSearchCV(base, grids[name], scoring='f1', cv=cv, refit=True, return_train_score=True)
    gscv.fit(X_train, y_train)
    best_pipes[name] = gscv.best_estimator_
    
    res = pd.DataFrame(gscv.cv_results_)
    cols = [c for c in res.columns if c.startswith('param_')] + ['mean_test_score', 'std_test_score', 'mean_train_score']
    gs_results[name] = res[cols].sort_values('mean_test_score', ascending=False).head(10)
```
</details>

<details>
<summary><b>Section 9: Model Selection</b></summary>

```python
for name, pipe in best_pipes.items():
    s = pipe.predict_proba(X_test)[:,1]
    scores_tuned[name] = s
    m, _ = evaluate_classifier(y_test, s, threshold=0.5)
    metrics_tuned[name] = m

summary_tuned = summarize_metrics(metrics_tuned)
best_name = summary_tuned.index[0]
```
</details>

<details>
<summary><b>Section 9: Diagnostic Plots</b></summary>

```python
y_score_best = scores_tuned[best_name]
m_best, y_pred_best = evaluate_classifier(y_test, y_score_best, threshold=0.5)

cm = confusion_matrix(y_test, y_pred_best)
plot_confusion_matrix(cm)
plot_roc_pr(y_test, y_score_best)
calibration_plot(y_test, y_score_best)
plot_probability_density(y_test, y_score_best)
```
</details>

<details>
<summary><b>Section 10: Threshold Optimization</b></summary>

```python
best_t, best_v, hist = find_best_threshold(y_test, y_score_best, metric='f1')
print(f'Best threshold (F1): {best_t:.4f} -> {best_v:.4f}')

plt.plot(hist['threshold'], hist['f1'])
plt.xlabel('Threshold'); plt.ylabel('F1'); 
plt.title(f'Threshold Sweep ({best_name})')
plt.show()

metrics_custom = {}
for name, s in scores_tuned.items():
    m, _ = evaluate_classifier(y_test, s, threshold=best_t)
    metrics_custom[name] = m
    
summary_custom = summarize_metrics(metrics_custom)
```
</details>

<details>
<summary><b>Section 11: Model Interpretation</b></summary>

```python
# Logistic Regression Coefficients
lr = best_pipes.get('LogisticRegression')
if lr is not None:
    coefs = lr.named_steps['clf'].coef_.ravel()
    coef_df = pd.DataFrame({'feature': feature_cols, 'coef': coefs})
    display(coef_df.assign(abs=lambda d: d['coef'].abs())
            .sort_values('abs', ascending=False).head(12))

# Decision Tree Permutation Importance
dt = best_pipes.get('DecisionTree')
if dt is not None:
    pi = permutation_importance(dt, X_test, y_test, n_repeats=10, random_state=RANDOM_STATE)
    display(pd.DataFrame({'feature': feature_cols, 'importance': pi.importances_mean})
            .sort_values('importance', ascending=False).head(12))
```
</details>

---

**Good luck! 🚀**